# Duelo dos 3 melhores modelos

Após a avaliação inicial dos seis modelos, foram selecionados os três modelos que apresentaram melhor desempenho para a classe 1, que representa internações prolongadas, levando em consideração principalmente métricas como recall, precision e F1-score. Essas métricas permitem avaliar tanto a capacidade do modelo de encontrar casos positivos quanto a qualidade das previsões realizadas.

Os modelos selecionados para esta etapa foram:

1. Regressão Logística com SMOTE;
2. Random Forest com SMOTE;
3. XGBoost com SMOTE.

A escolha pelas versões desses modelos ocorreu porque o balanceamento da classe minoritária durante o treinamento melhorou a capacidade de identificar internações prolongadas. Apesar de essa estratégia aumentar a quantidade de falsos positivos, ela ainda é relevante para o projeto, já que deixar de identificar uma internação prolongada pode ser mais prejudicial em um cenário de triagem hospitalar.

Nesta etapa, os três modelos passarão por ajustes básicos de hiperparâmetros. O objetivo ainda não é realizar uma busca exaustiva, mas verificar se pequenas alterações nas configurações principais de cada algoritmo conseguem melhorar o equilíbrio entre recall, precision e F1-score.

Ao final, será escolhido o modelo com melhor potencial para uma etapa posterior de refinamento, considerando não apenas a métrica isolada mais alta, mas o equilíbrio geral entre desempenho, interpretabilidade e capacidade de generalização.


In [1]:
from pathlib import Path
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score
)

Vou carregar as bases de treino e teste para trabalhar com os modelos.


In [2]:
pasta_saida = Path("dados/processed/splits")

x_treino_2020_2021 = pd.read_csv(pasta_saida / "x_treino_2020_2021.csv")
x_treino_2020_2021_smote = pd.read_csv(pasta_saida / "x_treino_2020_2021_smote.csv")
x_teste_2022 = pd.read_csv(pasta_saida / "x_teste_2022.csv")

y_treino_2020_2021 = pd.read_csv(pasta_saida / "y_treino_2020_2021.csv")["internacao_prolongada"]
y_treino_2020_2021_smote = pd.read_csv(pasta_saida / "y_treino_2020_2021_smote.csv")["internacao_prolongada"]
y_teste_2022 = pd.read_csv(pasta_saida / "y_teste_2022.csv")["internacao_prolongada"]


In [3]:
smote = SMOTE(random_state=42)

Nesta fase, vou usar os três modelos com melhor desempenho na etapa anterior, que são:

1. Regressão Logística com SMOTE: teve o melhor F1-score da tabela e o segundo melhor recall, sendo o modelo com melhor equilíbrio geral entre encontrar casos positivos e manter uma precision aceitável.

2. Random Forest com SMOTE: teve o segundo melhor F1-score e a melhor precision entre os modelos com SMOTE, apresentando desempenho mais equilibrado que o XGBoost com SMOTE.

3. XGBoost com SMOTE: teve o melhor recall da tabela, identificando mais casos positivos, mesmo com precision menor. Ainda assim, vale a pena avançar com ele, pois reduz bastante os falsos negativos.

Vou fazer um ajuste de hiperparâmetros que considero básico para cada modelo e, após observar seus respectivos desempenhos, levarei o melhor modelo para a última fase, onde realizarei um ajuste mais pesado de hiperparâmetros, visando obter melhor desempenho.


# Regressão Logística com SMOTE Ajustado

In [5]:
scaler = StandardScaler()

x_treino_2020_2021_padronizado = x_treino_2020_2021.copy()
x_teste_2022_padronizado = x_teste_2022.copy()

colunas_escalar = ["idade_anos","ano_internacao","mes_internacao",]

x_treino_2020_2021_padronizado[colunas_escalar] = scaler.fit_transform(x_treino_2020_2021[colunas_escalar])
x_teste_2022_padronizado[colunas_escalar] = scaler.transform(x_teste_2022[colunas_escalar])

x_treino_2020_2021_padronizado_smote, y_treino_2020_2021_smote = smote.fit_resample(x_treino_2020_2021_padronizado, y_treino_2020_2021)

In [6]:
modelo_reglog_ajustado = LogisticRegression(
    C=0.5,
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

modelo_reglog_ajustado.fit(x_treino_2020_2021_padronizado_smote, y_treino_2020_2021_smote)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.5
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

Agora vou fazer as previsões.


In [7]:
y_pred_reglog_smote_ajustado = modelo_reglog_ajustado.predict(x_teste_2022_padronizado)


Agora vou avaliar o modelo.


In [8]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2022, y_pred_reglog_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste_2022, y_pred_reglog_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste_2022, y_pred_reglog_smote_ajustado))
print('Recall:', recall_score(y_teste_2022, y_pred_reglog_smote_ajustado))
print('Precision:', precision_score(y_teste_2022, y_pred_reglog_smote_ajustado))
print('F1-score:', f1_score(y_teste_2022, y_pred_reglog_smote_ajustado))

Matriz de confusão:
[[129975  34338]
 [ 12225  31064]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.79      0.85    164313
           1       0.47      0.72      0.57     43289

    accuracy                           0.78    207602
   macro avg       0.69      0.75      0.71    207602
weighted avg       0.82      0.78      0.79    207602

Accuracy: 0.7757102532730898
Recall: 0.7175956940562268
Precision: 0.4749701843980306
F1-score: 0.5716020645683635


O modelo identificou corretamente 31.064 internações prolongadas no ano de 2022, enquanto deixou de identificar 12.225 casos reais da classe 1. O recall foi de aproximadamente 71,76%, indicando que o modelo conseguiu detectar uma parcela relevante das internações prolongadas.

A precision foi de aproximadamente 47,50%, mostrando que, entre os casos classificados como internação prolongada, pouco menos da metade realmente pertencia à classe 1. Já o F1-score foi de aproximadamente 57,16%, indicando um equilíbrio entre recall e precision.

Sendo assim, de forma geral, o modelo manteve boa capacidade de identificar internações prolongadas, porém ainda apresentou uma quantidade considerável de falsos positivos. O ajuste com `C=0.5` tornou o modelo um pouco mais regularizado, buscando controlar a complexidade sem reduzir excessivamente o recall.


# Random Forest com SMOTE ajustado

In [11]:
modelo_random_forest_smote_ajustado = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

modelo_random_forest_smote_ajustado.fit(x_treino_2020_2021_smote, y_treino_2020_2021_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

Agora vou fazer as previsões.


In [12]:
y_pred_random_forest_smote_ajustado = modelo_random_forest_smote_ajustado.predict(x_teste_2022)

Agora vou avaliar o modelo.


In [13]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2022, y_pred_random_forest_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste_2022, y_pred_random_forest_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste_2022, y_pred_random_forest_smote_ajustado))
print('Recall:', recall_score(y_teste_2022, y_pred_random_forest_smote_ajustado))
print('Precision:', precision_score(y_teste_2022, y_pred_random_forest_smote_ajustado))
print('F1-score:', f1_score(y_teste_2022, y_pred_random_forest_smote_ajustado))

Matriz de confusão:
[[113061  51252]
 [ 11864  31425]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.69      0.78    164313
           1       0.38      0.73      0.50     43289

    accuracy                           0.70    207602
   macro avg       0.64      0.71      0.64    207602
weighted avg       0.80      0.70      0.72    207602

Accuracy: 0.6959759539888826
Recall: 0.7259349950333803
Precision: 0.3800936173300918
F1-score: 0.4989441595351127


O modelo identificou corretamente 31.425 internações prolongadas no ano de 2022, deixando de identificar 11.864 casos reais da classe 1. O recall foi de aproximadamente 72,59%, mostrando que o modelo conseguiu detectar uma parcela relevante das internações prolongadas.

Porém, a precision foi de aproximadamente 38,01%, indicando que, entre os casos classificados como internação prolongada, uma parte considerável não pertencia à classe 1.

Isso mostra que o modelo se tornou bastante sensível à classe de interesse, mas ao custo de um número elevado de falsos positivos. Ele classificou 51.252 internações não prolongadas como prolongadas, reduzindo, assim, sua precision e seu F1-score.


# XGBoost com SMOTE Ajustado

In [16]:
modelo_xgboost_smote_ajustado = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.07,
    min_child_weight=5,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    reg_alpha=0,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

modelo_xgboost_smote_ajustado.fit(x_treino_2020_2021_smote, y_treino_2020_2021_smote)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


Agora vou fazer as previsões.


In [17]:
y_pred_xgboost_smote_ajustado = modelo_xgboost_smote_ajustado.predict(x_teste_2022)

Agora vou avaliar o modelo.


In [18]:
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2022, y_pred_xgboost_smote_ajustado))

print('\nClassification Report:')
print(classification_report(y_teste_2022, y_pred_xgboost_smote_ajustado))

print('Accuracy:', accuracy_score(y_teste_2022, y_pred_xgboost_smote_ajustado))
print('Recall:', recall_score(y_teste_2022, y_pred_xgboost_smote_ajustado))
print('Precision:', precision_score(y_teste_2022, y_pred_xgboost_smote_ajustado))
print('F1-score:', f1_score(y_teste_2022, y_pred_xgboost_smote_ajustado))

Matriz de confusão:
[[124741  39572]
 [ 12715  30574]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.76      0.83    164313
           1       0.44      0.71      0.54     43289

    accuracy                           0.75    207602
   macro avg       0.67      0.73      0.68    207602
weighted avg       0.81      0.75      0.77    207602

Accuracy: 0.7481382645639252
Recall: 0.7062764212617524
Precision: 0.4358623442534143
F1-score: 0.5390576100850707


O modelo identificou corretamente 30.574 internações prolongadas no ano de 2022, enquanto deixou de identificar 12.715 casos reais da classe 1. O recall da classe 1 foi de aproximadamente 70,63%, mostrando que o modelo conseguiu detectar uma parcela relevante das internações prolongadas.

A precision foi de aproximadamente 43,59%, indicando que, entre os casos classificados como internação prolongada, menos da metade realmente pertencia à classe 1. O F1-score foi de aproximadamente 53,91%, mostrando um desempenho intermediário entre recall e precision.

Mesmo o modelo tendo mantido boa capacidade de identificar internações prolongadas, ele ainda gerou uma quantidade elevada de falsos positivos, com 39.572 internações não prolongadas classificadas incorretamente como prolongadas.

De forma geral, ele apresentou desempenho intermediário, conseguindo detectar boa parte das internações prolongadas, apesar de não apresentar o melhor equilíbrio entre recall e precision nesta etapa de ajuste básico.


# Conclusão

É possível observar que:

1. A Regressão Logística ajustada teve o melhor desempenho geral entre os modelos avaliados. Apesar de não ter obtido o maior recall, apresentou o maior F1-score e a melhor precision, mantendo ainda uma boa capacidade de identificar internações prolongadas. Isso indica que o modelo conseguiu equilibrar melhor a detecção da classe de interesse com uma quantidade menor de falsos positivos.

2. O Random Forest ajustado apresentou recall elevado, identificando uma quantidade relevante de internações prolongadas. Porém, sua precision foi a mais baixa entre os três modelos, indicando que ele classificou muitos casos não prolongados como prolongados. Como consequência, seu F1-score foi inferior ao dos demais modelos, mostrando menor equilíbrio entre recall e precision.

3. O XGBoost ajustado apresentou o maior recall entre os modelos avaliados, sendo o que mais identificou corretamente internações prolongadas. Porém, sua precision foi inferior à da Regressão Logística ajustada, indicando maior número de falsos positivos. Ainda assim, seu desempenho foi superior ao do Random Forest ajustado em F1-score, tornando-o uma opção interessante para ajustes posteriores.

Em suma, a Regressão Logística ajustada foi o melhor modelo nesta etapa, levando em conta o equilíbrio entre recall, precision e F1-score. O XGBoost ajustado teve alto recall e apresenta maior possibilidade de ajustes de hiperparâmetros, permanecendo como um bom candidato para a etapa de ajustes mais finos. Já o Random Forest ajustado teve desempenho pior em equilíbrio geral, principalmente devido à baixa precision e ao elevado número de falsos positivos.
